### Overview: 
This code implements one of the multiple ways of multi-model RAG. It extracts and processes text and images from PDFs, utilizing a multi-modal Retrieval-Augmented Generation (RAG) system for summarizing and retrieving content for question answering.

### Key Components:
   - **PyMuPDF**: For extracting text and images from PDFs.
   - **Gemini 1.5-flash model**: To summarize images and tables.
   - **Cohere Embeddings**: For embedding document splits.
   - **Chroma Vectorstore**: To store and retrieve document embeddings.
   - **LangChain**: To orchestrate the retrieval and generation pipeline.

### Diagram:
   <img src="../images/multi_model_rag_with_captioning.svg" alt="Reliable-RAG" width="300">

### Motivation: 
Efficiently summarize complex documents to facilitate easy retrieval and concise responses for multi-modal data.

### Method Details:
   - Text and images are extracted from the PDF using PyMuPDF.
   - Summarization is performed on extracted images and tables using Gemini.
   - Embeddings are generated via Cohere for storage in Chroma.
   - A similarity-based retriever fetches relevant sections based on the query.

### Benefits:
   - Simplified retrieval from complex, multi-modal documents.
   - Streamlined Q&A process for both text and images.
   - Flexible architecture for expanding to more document types.

### Implementation:
   - Documents are split into chunks with overlap using a text splitter.
   - Summarized text and image content are stored as vectors.
   - Queries are handled by retrieving relevant document segments and generating concise answers.

### Summary: 
The project enables multi-modal document processing and retrieval, providing concise, relevant responses by combining state-of-the-art LLMs and vector-based retrieval systems.

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.


In [23]:
import fitz  # PyMuPDF
from PIL import Image
import io
import os
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

load_dotenv()

False

### Data Extraction

In [24]:
text_data = []
img_data = []

In [25]:
with fitz.open('data/attention_is_all_you_need.pdf') as pdf_file:
    # Create a directory to store the images
    if not os.path.exists("extracted_images"):
        os.makedirs("extracted_images")

    # Loop through every page in the PDF
    for page_number in range(len(pdf_file)):
        page = pdf_file[page_number]
        
        # Get the text on page
        text = page.get_text().strip()
        text_data.append({"response": text, "name": page_number+1})
        # Get the list of images on the page
        images = page.get_images(full=True)

        # Loop through all images found on the page
        for image_index, img in enumerate(images, start=0):
            xref = img[0]  # Get the XREF of the image
            base_image = pdf_file.extract_image(xref)  # Extract the image
            image_bytes = base_image["image"]  # Get the image bytes
            image_ext = base_image["ext"]  # Get the image extension
            
            # Load the image using PIL and save it
            image = Image.open(io.BytesIO(image_bytes))
            image.save(f"extracted_images/image_{page_number+1}_{image_index+1}.{image_ext}")

### Image Captioning

In [26]:
from dashscope import MultiModalConversation

for img in os.listdir("extracted_images"):
    image = f"extracted_images/{img}"
    messages = [
        {
            "role": "user",
            "content": [
                {"image": image},
                {"text": "You are an assistant tasked with summarizing tables, images and text for retrieval. \
    These summaries will be embedded and used to retrieve the raw text or table elements \
    Give a concise summary of the table or text that is well optimized for retrieval. Table or text or image:"}
            ]
        }
    ]

    response = MultiModalConversation.call(
        # 若没有配置环境变量，请用百炼API Key将下行替换为：api_key="sk-xxx",
        api_key=os.getenv('DASHSCOPE_API_KEY'),
        model="qwen3-vl-flash",
        messages=messages,
        stream=True,
        # enable_thinking 参数开启思考过程
        enable_thinking=True,
        # thinking_budget 参数设置最大推理过程 Token 数
        thinking_budget=81920,

    )

    # 定义完整思考过程
    reasoning_content = ""
    # 定义完整回复
    answer_content = ""
    # 判断是否结束思考过程并开始回复
    is_answering = False

    # print("=" * 20 + "思考过程" + "=" * 20)

    for chunk in response:
        # 如果思考过程与回复皆为空，则忽略
        message = chunk.output.choices[0].message
        reasoning_content_chunk = message.get("reasoning_content", None)
        if (chunk.output.choices[0].message.content == [] and
            reasoning_content_chunk == ""):
            pass
        else:
            # 如果当前为思考过程
            if reasoning_content_chunk != None and chunk.output.choices[0].message.content == []:
                # print(chunk.output.choices[0].message.reasoning_content, end="")
                # reasoning_content += chunk.output.choices[0].message.reasoning_content
                pass
            # 如果当前为回复
            elif chunk.output.choices[0].message.content != []:
                if not is_answering:
                    print("\n" + "=" * 20 + "完整回复" + "=" * 20)
                    is_answering = True
                print(chunk.output.choices[0].message.content[0]["text"], end="")
                answer_content += chunk.output.choices[0].message.content[0]["text"]
    img_data.append({"response": answer_content, "name": img})




====================完整回复====================
Self-attention computation flow: Input Q and K undergo MatMul → Scale → (optional Mask) → SoftMax → MatMul with V → Output.
====================完整回复====================
Scaled Dot-Product Attention module with input Q (Query), K (Key), V (Value). Each input passes through a Linear layer, outputs fed into Scaled Dot-Product Attention, followed by Concat (concatenation), then a final Linear layer.
====================完整回复====================
Transformer model architecture diagram depicting encoder (left) and decoder (right) blocks. Both contain *Nx* stacked layers, each with multi - head attention, feed forward, and add & norm. Encoder processes inputs via input embedding + positional encoding. Decoder uses output embedding + positional encoding (shifted right), employs masked multi - head attention, and outputs probabilities via linear + softmax.

### Vectostore

In [27]:
from helper_functions import DashScopeEmbeddings

In [28]:
# Set embeddings
embedding_model = DashScopeEmbeddings()

# Load the document
docs_list = [Document(page_content=text['response'], metadata={"name": text['name']}) for text in text_data]
img_list = [Document(page_content=img['response'], metadata={"name": img['name']}) for img in img_data]

# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400, chunk_overlap=50
)

doc_splits = text_splitter.split_documents(docs_list)
img_splits = text_splitter.split_documents(img_list)

In [29]:
# Add to vectorstore
vectorstore = Chroma.from_documents(
    documents=doc_splits + img_splits, # adding the both text and image splits
    collection_name="multi_model_rag",
    embedding=embedding_model,
)

retriever = vectorstore.as_retriever(
                search_type="similarity",
                search_kwargs={'k': 1}, # number of documents to retrieve
            )

### Query

In [30]:
query = "What is the BLEU score of the Transformer (base model)?"

In [31]:
docs = retriever.invoke(query)

In [32]:
docs

[Document(metadata={'name': 8}, page_content='Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the\nEnglish-to-German and English-to-French newstest2014 tests at a fraction of the training cost.\nModel\nBLEU\nTraining Cost (FLOPs)\nEN-DE\nEN-FR\nEN-DE\nEN-FR\nByteNet [18]\n23.75\nDeep-Att + PosUnk [39]\n39.2\n1.0 · 1020\nGNMT + RL [38]\n24.6\n39.92\n2.3 · 1019\n1.4 · 1020\nConvS2S [9]\n25.16\n40.46\n9.6 · 1018\n1.5 · 1020\nMoE [32]\n26.03\n40.56\n2.0 · 1019\n1.2 · 1020\nDeep-Att + PosUnk Ensemble [39]\n40.4\n8.0 · 1020\nGNMT + RL Ensemble [38]\n26.30\n41.16\n1.8 · 1020\n1.1 · 1021\nConvS2S Ensemble [9]\n26.36\n41.29\n7.7 · 1019\n1.2 · 1021\nTransformer (base model)\n27.3\n38.1\n3.3 · 1018\nTransformer (big)\n28.4\n41.8\n2.3 · 1019\nResidual Dropout\nWe apply dropout [33] to the output of each sub-layer, before it is added to the\nsub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the')]

### Output

In [33]:
from langchain_core.output_parsers import StrOutputParser
from langchain_deepseek import ChatDeepSeek
# Prompt
system = """You are an assistant for question-answering tasks. Answer the question based upon your knowledge. 
Use three-to-five sentences maximum and keep the answer concise."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved documents: \n\n <docs>{documents}</docs> \n\n User question: <question>{question}</question>"),
    ]
)

# LLM
llm = ChatDeepSeek(
            model="deepseek-chat",
            temperature=0,
        )

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"documents":docs[0].page_content, "question": query})
print(generation)

The Transformer (base model) achieves a BLEU score of 27.3 for English-to-German (EN-DE) translation and 38.1 for English-to-French (EN-FR) translation. These scores are reported on the newstest2014 benchmarks.


![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--multi-model-rag-with-captioning)